In [34]:
%pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [35]:
import numpy as np

print(np.__version__)

2.5.4


In [36]:
# C-like structs.
import struct

# Math.
import math

# Problem 1: Representing SHA-256 Data

Before implementing SHA-256, investigate how the input, output, and intermediate data can be represented in Python.

Explain how you would represent:

32-bit words,

sequences of 32-bit words,

input messages,

512-bit message blocks,

the final 256-bit hash value,

big-endian integers.

For each representation, explain why it is appropriate and demonstrate your choices with small Python examples.

## 32-bit words

A 32-bit word contains 32 bits or 4 bytes.

Bits: 1010 0001 0000 0011 1111 1110 0010 0011
Hexadecimal: a103fe23

According to the NIST Secure Hash Standard ([Section 3.1](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf)):
>"An integer between 0 and $2^{32} - 1$ inclusive may be represented as a 32-bit word.  The least significant four bits of the integer are represented by the right-most hex digit of the word representation. For example, the integer 291=28 + 25 + 21 + 20=256+32+2+1 is represented by the hex word “00000123” If Z is an integer, 0 ≤ Z < $2^{64}$, then Z=$2^{32}$X + Y, where 0 ≤ X < $2^{32}$ and 0 ≤ Y < $2^{32}$.  Since X and Y can be represented as 32-bit words x and y, respectively, the integer Z can be represented as the pair of words (x, y).  This property is used for SHA-1, SHA224 and SHA-256." 


In [37]:
# create a variable for 32-bit word
# the smallest valid unsigned word in int
word = 0

print("Value:", word)

# show word in binary string
print("Binary:", bin(word)) 

# show size of word
print("Number of significant bits:", word.bit_length())
print("32-bit representation:", format(word, "032b"))

# the biggest valid unsigned word in int
word2 = 2**32 - 1

print("Value:", word2)

# show word in binary string
print("Binary:", bin(word2)) 

# show size of word
print("Number of significant bits:", word2.bit_length())
print("32-bit representation:", format(word2, "032b"))

# the first value out of range
word2 = 2**32 + 1

print("Value:", word2)

# show word in binary string
print("Binary:", bin(word2)) 

# show size of word
print("Number of significant bits:", word2.bit_length())
print("32-bit representation:", format(word2, "032b"))

Value: 0
Binary: 0b0
Number of significant bits: 0
32-bit representation: 00000000000000000000000000000000
Value: 4294967295
Binary: 0b11111111111111111111111111111111
Number of significant bits: 32
32-bit representation: 11111111111111111111111111111111
Value: 4294967297
Binary: 0b100000000000000000000000000000001
Number of significant bits: 33
32-bit representation: 100000000000000000000000000000001


### Why did you choose Python's int?
I chose python's int as an int between 0 and $2^{32} - 1$ inclusive may be represented as a 32-bit word. However, Python integers have arbitrary precision, so using int alone does not restrict a value to 32 bits.

### What is bit_length()?
bit_length() tells me the number of significant bits in the given data type, without the 0s. For example, the integer 5 has a bit length of 3 because its binary representation is 101.

### Why format(word, "032b") is useful.
This auto formats the word into a minimum of 32 bits. It displays an integer in binary (b), with a minimum width of 32 characters (32) and leading zeros when needed (0).

### Why formatting alone doesn't guarantee that a number fits within 32 bits.
Formatting alone does not guarantee a value fits within 32 bits because Python's format width is a minimum rather than a maximum, thus python can still display a value with 33 bits as it meets the minimum requirement. Therefore, we must also check that the integer lies between 0 and $2^{32} - 1$ inclusive.

## Sequences of 32 bit words

A sequence of 32 bit words is an ordered collection --> can be used for operations like addition, multiplication etc

Since 32-bit words can be represented as integers, a Python list of integers is a suitable representation for a sequence of words.

 *A list is suitable because it stores multiple values in order and allows me to iterate through each word easily. Python lists are flexible and do not require an additional library.*
 
However, each integer must be checked to ensure that it falls within the unsigned 32-bit range, from 0 to $(2^{32}-1)$

In [38]:
# since one 32-bit word is best represented as an int
word1 = 5;
word2 = 10;
word3 = 15;

#create the collection of 32 bit words
sequence = [word1, word2, word3];

print(sequence);

for word in sequence:
    if not 0 <= word <= 0xFFFFFFFF: #0xFFFFFFFF is a hex num where all 32 bits is set to 1
        raise ValueError("Value is outside the 32-bit unsigned range")

    print(format(word, "032b")) #print as a 32 bit binary word

[5, 10, 15]
00000000000000000000000000000101
00000000000000000000000000001010
00000000000000000000000000001111


In [39]:
# considering out of range values 
word = 2**32;

try:
    if not 0 <= word <= 0xFFFFFFFF:
        raise ValueError("Value is outside the 32-bit unsigned range")

    print(format(word, "032b"))

except ValueError as error:
    print("Validation error:", error)

Validation error: Value is outside the 32-bit unsigned range


### Why a for loop?
The for loop iterates through each element in the list and prints its 32-bit binary representation. Hence, when I add another word to the list, the loop processes it automatically, so I don't need to change the loop. This makes it easier to use.

## Input Messages

User prompt: A prompt such as "Enter a message:" asks the user to type something.

Input message: The actual text the user enters, such as "hello", which will be hashed.

**Does the message need a dynamic data type, since we do not know the length of the user's input in advance?**

*Python's bytes type can represent sequences of different lengths, so it is suitable for storing input messages of varying sizes.*

According to the NIST Secure Hash Standard ([Section 3.1](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf)):

>"For SHA-1, SHA-224 and SHA-256, each message block has 512 bits, which are represented as a sequence of sixteen 32-bit words."

This means that each message block contains 512 bits, arranged into 16 words of 32 bits each. 8 bits = 1 byte, hence each block conatins 64 bytes

*I think Python's bytes type is a suitable representation for input messages because it stores a sequence of byte values, each ranging from 0 to 255. If the input is text, it can first be converted into bytes using an encoding such as UTF-8.*

In [40]:
#ordinary string
message = "happy days"
print(message)
print("Message type:", type(message))

#convert string to bytes
my_word = "hello"                 # str
my_bytes = my_word.encode("utf-8")  # bytes

print(my_bytes)
print("Message type:", type(my_bytes))
print("Message length:", len(my_bytes))

happy days
Message type: <class 'str'>
b'hello'
Message type: <class 'bytes'>
Message length: 5


This is useful because SHA-256 processes message data as bits, so text must be converted into bytes before it can be processed by the algorithm. Using Python's bytes type provides a suitable representation of the input message.

## 512-bit message blocks



The chunks of data by SHA-256 that are represented as a sequence of sixteen 32-bit words.

512 bits = 64 bytes per block.

According to the NIST Secure Hash Standard ([Section 5.1.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf)):

>"For SHA-1, SHA-224 and SHA-256, the message and its padding are parsed into N 512-bit blocks ... the 512 bits of the input block may be expressed as sixteen 32-bit word"

As established earlier for sequences of 32-bit words, this is best represented by a list. Therefore, a 512-bit message block could also be represented by a list containing 16 integers, with each integer representing one 32-bit word. 

*A list is convenient because it allows individual words to be accessed and used in the operations performed by SHA-256. The code below demonstrates this representation by creating a list of 16 integers and checking the number of words and total number of bits.*

In [41]:
# since one 32-bit word is best represented as an int, create 16 words to represent a 512-bit message
sequence = []

for i in range(1, 17):
    sequence.append(i * 5)

print(sequence)
print ("Number of words in sequence:", len(sequence))
print("Number of bits in sequence:", len(sequence) * 32)

[5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80]
Number of words in sequence: 16
Number of bits in sequence: 512


##  Final 256-bit hash value

According to the NIST Secure Hash Standard (Section 6.2):
>"The final result of SHA-256 is a 256-bit message digest"

This means that the final 256-bit hash value is the output SHA-256 produces after hashing the input message, by processing the 512-message blocks including the padding.

*Hence, Python's bytes type is a suitable representation because it stores a sequence of byte values, and each byte contains 8 bits.*

In [42]:
import hashlib #provides hashing algorithms like SHA-256

#create a bytes object
message = b"hello"
digest = hashlib.sha256(message).digest()

print("Digest:", digest)
print("Digest type:", type(digest))
print("Digest length in bytes:", len(digest))
print("Digest length in bits:", len(digest) * 8)
print("Hexadecimal digest:", digest.hex())
print("Hexadecimal length:", len(digest.hex()))

Digest: b',\xf2M\xba_\xb0\xa3\x0e&\xe8;*\xc5\xb9\xe2\x9e\x1b\x16\x1e\\\x1f\xa7B^s\x043b\x93\x8b\x98$'
Digest type: <class 'bytes'>
Digest length in bytes: 32
Digest length in bits: 256
Hexadecimal digest: 2cf24dba5fb0a30e26e83b2ac5b9e29e1b161e5c1fa7425e73043362938b9824
Hexadecimal length: 64


### Why not an integer?
When you represent a hash as an integer, leading zeros can disappear as it does not preserved in its usual numerical representation, which can make its displayed bit length shorter than 256 bits. Representing the digest as bytes keeps its fixed length of 32 bytes (256 bits), including any leading zero bytes.

## Big-endian integers

According to the NIST Secure Hash Standard (Section 3.1):
>"Throughout this specification, the “big-endian” convention is used when expressing both 32- and 64-bit words, so that within each word, the most significant bit is stored in the left-most bit position."

Applying this concept by replacing bits with bytes, a Big-endian representation is a way of arranging bytes when representing an integer. The most significant byte (the byte with the greatest place value) comes first, followed by the remaining bytes in descending significance.

Python's int type can represent the numerical value, while bytes can represent its bytes in a specific order. The following experiment demonstrates how Python converts an integer into big-endian bytes and then converts it back into an integer.

In [43]:
number = 0x12345678

# convert integer to bytes using big-endian byte order
data = number.to_bytes(4, byteorder="big")

print("Integer:", number)
print("Big-endian bytes:", data)
print("Hexadecimal representation:", data.hex())
print("Back to integer:", int.from_bytes(data, "big"))

Integer: 305419896
Big-endian bytes: b'\x124Vx'
Hexadecimal representation: 12345678
Back to integer: 305419896


# Problem 2: SHA-256 Bitwise Operations

Implement the Boolean and bitwise functions used by SHA-256:

1. `Ch(x, y, z)`
2. `Maj(x, y, z)`
3. `Sigma0(x)` — written as $\Sigma_0^{256}(x)$ in the standard
4. `Sigma1(x)` — written as $\Sigma_1^{256}(x)$ in the standard
5. `sigma0(x)` — written as $\sigma_0^{256}(x)$ in the standard
6. `sigma1(x)` — written as $\sigma_1^{256}(x)$ in the standard

Use NumPy so that the values involved in these operations are represented as 32-bit unsigned integers.

According to the NIST Secure Hash Standard ([Section 4.1.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf)):
>"SHA-224 and SHA-256 both use six logical functions, where each function operates on 32-bit words, which are represented as x, y, and z. The result of each function is a new 32-bit word."

## Ch(x, y, z)
**What is a Ch() function?**

It is a choose function in SHA-256 where it uses x to choose between y and z, bit by bit.

According to the NIST Secure Hash Standard ([Section 4.1.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf)):
$$
\operatorname{Ch}(x,y,z) = (x \land y) \oplus (\neg x \land z)
$$

**What do the symbols mean?**

According to the NIST Secure Hash Standard ([Section 4.1.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf)):
- $\land$ — Bitwise AND operation.
- $\oplus$ — Bitwise XOR (“exclusive-OR”) operation.
- $\neg$ — Bitwise complement operation.

**How do I apply these symbols to the given formula, step by step?**

- x $\land$ y -> x AND y 
- $\neg$ x $\land$ z -> complement x AND z
- $(x \land y) \oplus (\neg x \land z)$ -> (x AND y) XOR (omplement x AND z)


**Relevant Bitwise Truth Tables**
    
<img src="images/TruthTables.jpeg" width="500">

[Representations of Logic Operations, Truth tables and their relation to logic circuits and equations](https://www.realdigital.org/doc/e127ebfa82dbc904b5c0dac5d1adce8e)

Hence, for $\operatorname{Ch}(1,1,0)$:

$$
\begin{aligned}
\operatorname{Ch}(1,1,0)
&= (1 \land 1) \oplus (\neg 1 \land 0) \\
&= (1 \land 1) \oplus (0 \land 0) \\
&= 1 \oplus 0 \\
&= 1
\end{aligned}
$$


## Maj(x, y, z)

## Sigma0(x) — written as $\Sigma_0^{256}(x)$ in the standard

## Sigma1(x) — written as $\Sigma_1^{256}(x)$ in the standard

## sigma0(x) — written as $\sigma_0^{256}(x)$ in the standard

## sigma1(x) — written as $\sigma_1^{256}(x)$ in the standard